<a href="https://colab.research.google.com/github/behraj/FIcsR/blob/main/FIcsR_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Train/Test Split

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.distributions.kl import kl_divergence
from tqdm import tqdm

# Hyperparameters
batch_size = 64
num_epochs = 10
learning_rate = 1e-3
lambda_reg = 0.1  # Regularization term scaling for KL divergence

# Variational Inference Parameters (using Monte Carlo Dropout)
dropout_prob = 0.2

# MNIST Dataset
transform = transforms.Compose([transforms.ToTensor()])
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Data loaders
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

# Model definition with Dropout for Monte Carlo sampling
class BayesianNN(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(BayesianNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        return self.fc3(x)

# Custom function to compute KL divergence
def kl_regularization(q_logits, p_logits):
    q_dist = torch.distributions.Categorical(logits=q_logits)
    p_dist = torch.distributions.Categorical(logits=p_logits)
    return kl_divergence(q_dist, p_dist).mean()

# Training function with VI-based regularization
def train_with_vi(model, train_loader, optimizer, criterion, lambda_reg):
    model.train()
    p_prior_logits = None

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()

            # Forward pass with Monte Carlo Dropout
            q_logits = model(data)
            loss = criterion(q_logits, target)

            # KL Divergence with previous batch (simulating prior adaptation)
            if p_prior_logits is not None and p_prior_logits.shape == q_logits.shape:
                kl_loss = kl_regularization(q_logits, p_prior_logits)
                loss += lambda_reg * kl_loss

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(train_loader):.4f}")

        # Set p_prior_logits for the next epoch
        with torch.no_grad():
            p_prior_logits = q_logits.clone().detach()

# Evaluation function
def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for data, target in data_loader:
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    avg_loss = total_loss / len(data_loader)
    accuracy = correct / len(data_loader.dataset)
    return avg_loss, accuracy

# Main script
model = BayesianNN(dropout_prob=dropout_prob)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Train and evaluate with VI adaptation
print("Training with Variational Inference (VI) for Covariate Shift Adaptation...")
train_with_vi(model, train_loader, optimizer, criterion, lambda_reg)

# Evaluate on the test set
test_loss, test_accuracy = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy * 100:.2f}%")


### Batches 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torch.distributions.kl import kl_divergence
from tqdm import tqdm
import numpy as np

# Hyperparameters
batch_size = 64
num_epochs = 10
learning_rate = 1e-3
lambda_reg = 0.1  # Regularization term scaling for KL divergence

# Variational Inference Parameters (using Monte Carlo Dropout)
dropout_prob = 0.2
num_batches = 2

# MNIST Dataset
transform = transforms.Compose([transforms.ToTensor()])
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Model definition with Dropout for Monte Carlo sampling
class BayesianNN(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(BayesianNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        return self.fc3(x)

# Custom function to compute KL divergence
def kl_regularization(q_logits, p_logits):
    q_dist = torch.distributions.Categorical(logits=q_logits)
    p_dist = torch.distributions.Categorical(logits=p_logits)
    return kl_divergence(q_dist, p_dist).mean()

# Splitting MNIST into sequential batches (simulate covariate shift by digit class)
def create_batches(dataset, num_batches):
    indices = np.arange(len(dataset))
    np.random.shuffle(indices)
    batch_size = len(dataset) // num_batches
    return [Subset(dataset, indices[i*batch_size:(i+1)*batch_size]) for i in range(num_batches)]

# Training function with VI-based adaptation across batches
# Updated train_with_vi function with shape check for KL divergence
def train_with_vi(model, batches, optimizer, criterion, lambda_reg):
    model.train()
    p_prior_logits = None

    for i, batch in enumerate(batches):
        data_loader = DataLoader(batch, batch_size=batch_size, shuffle=True)
        for epoch in range(num_epochs):
            epoch_loss = 0.0
            for data, target in data_loader:
                optimizer.zero_grad()

                # Forward pass with Monte Carlo Dropout
                q_logits = model(data)
                loss = criterion(q_logits, target)

                # KL Divergence with previous batch (simulating prior adaptation)
                if p_prior_logits is not None and p_prior_logits.shape == q_logits.shape:
                    kl_loss = kl_regularization(q_logits, p_prior_logits)
                    loss += lambda_reg * kl_loss

                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            # print(f"Batch {i+1}/{len(batches)}, Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(data_loader):.4f}")

        # Set p_prior_logits for the next batch if batch size matches
        with torch.no_grad():
            p_prior_logits = q_logits.clone().detach()



# Evaluation function
def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for data, target in data_loader:
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    avg_loss = total_loss / len(data_loader)
    accuracy = correct / len(data_loader.dataset)
    return avg_loss, accuracy

# Main script
model = BayesianNN(dropout_prob=dropout_prob)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Fragment dataset into batches
train_batches = create_batches(mnist_train, num_batches)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

# Train and evaluate with VI adaptation
print("Training with Variational Inference (VI) for Covariate Shift Adaptation...")
train_with_vi(model, train_batches, optimizer, criterion, lambda_reg)

# Evaluate on the test set
test_loss, test_accuracy = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy * 100:.2f}%")
